In [2]:
%pip install -q opencc-python-reimplemented sentencepiece

Note: you may need to restart the kernel to use updated packages.


In [3]:
import os, json, random, unicodedata
from datasets import load_dataset
from opencc import OpenCC
import sentencepiece as spm
os.chdir("/workspace/shreshth")

SEED, CHARS_PER_LANG, VOCAB = 42, 20_000_000, 32000
LANGS = ["en", "hi", "zh", "ar"]
t2s = OpenCC("t2s")

def norm(s):
    return unicodedata.normalize("NFC", s or "").strip()

def take_until(lines, budget):
    out, total = [], 0
    for line in lines:
        if total >= budget:
            break
        out.append(line); total += len(line)
    return out

def jsonl_field(path, field):
    with open(path, encoding="utf-8") as f:
        for l in f:
            yield json.loads(l)[field]

def wiki_lines(config):
    seen, is_zh = set(), config.endswith(".zh")
    for art in load_dataset("wikimedia/wikipedia", config, split="train", streaming=True):
        for line in art["text"].split("\n"):
            line = norm(line)
            if is_zh:
                line = t2s.convert(line)
            if 20 <= len(line) <= 1000 and line not in seen:
                seen.add(line)
                yield line

half = CHARS_PER_LANG // 2
text = {
    # each language: half from our translation data (its domain), half from Wikipedia (general vocabulary)
    "en": take_until(jsonl_field("data/en-hi/train.jsonl", "en"), half) + take_until(wiki_lines("20231101.en"), half),
    "hi": take_until(jsonl_field("data/en-hi/train.jsonl", "tgt"), half) + take_until(wiki_lines("20231101.hi"), half),
    "zh": take_until(jsonl_field("data/en-zh/train.jsonl", "tgt"), half) + take_until(wiki_lines("20231101.zh"), half),
    "ar": take_until(jsonl_field("data/en-ar/train.jsonl", "tgt"), half) + take_until(wiki_lines("20231101.ar"), half),
}
lines = []
for l in LANGS:
    print(f"{l}: {len(text[l]):,} lines, {sum(map(len, text[l])):,} chars")
    lines += text[l]
random.Random(SEED).shuffle(lines)
os.makedirs("tokenizer_v4", exist_ok=True)
with open("data/tok_train_v4.txt", "w", encoding="utf-8") as f:
    f.write("\n".join(lines) + "\n")
del lines, text

spm.SentencePieceTrainer.train(
    input="data/tok_train_v4.txt", model_prefix="tokenizer_v4/spm", model_type="unigram",
    vocab_size=VOCAB, character_coverage=0.9995, byte_fallback=True, normalization_rule_name="identity",
    user_defined_symbols=["<2hi>", "<2zh>", "<2ar>"], pad_id=0, unk_id=1, bos_id=2, eos_id=3,
    input_sentence_size=4_000_000, shuffle_input_sentence=True, max_sentence_length=4096,
)

sp = spm.SentencePieceProcessor(model_file="tokenizer_v4/spm.model")
print("\nchars/token on FLORES dev:")
for l in LANGS:
    sents = [json.loads(x)["text"] for x in open(f"data/flores/dev.{l}.jsonl", encoding="utf-8")]
    print(f"  {l}: {sum(map(len, sents)) / sum(len(i) for i in sp.encode(sents)):.2f}")
print("\nFLORES dev[0] pieces:")
for l in LANGS:
    s = json.loads(open(f"data/flores/dev.{l}.jsonl", encoding="utf-8").readline())["text"]
    print(f"  {l}: {sp.encode(s, out_type=str)}")

README.md:   0%|          | 0.00/131k [00:00<?, ?B/s]

Resolving data files:   0%|          | 0/41 [00:00<?, ?it/s]

en: 160,979 lines, 20,000,313 chars
hi: 168,705 lines, 20,000,181 chars
zh: 485,341 lines, 20,000,187 chars
ar: 237,880 lines, 20,000,610 chars


I0000 00:00:1791343912.243303    3429 sentencepiece_trainer.cc:105] Starts training with : 
trainer_spec {
  input: data/tok_train_v4.txt
  input_format: 
  model_prefix: tokenizer_v4/spm
  model_type: UNIGRAM
  vocab_size: 32000
  self_test_sample_size: 0
  character_coverage: 0.9995
  input_sentence_size: 4000000
  shuffle_input_sentence: 1
  seed_sentencepiece_size: 1000000
  shrinking_factor: 0.75
  max_sentence_length: 4096
  num_threads: 16
  num_sub_iterations: 2
  max_sentencepiece_length: 16
  split_by_unicode_script: 1
  split_by_number: 1
  split_by_whitespace: 1
  split_digits: 0
  pretokenization_delimiter: 
  treat_whitespace_as_suffix: 0
  allow_whitespace_only_pieces: 0
  user_defined_symbols: <2hi>
  user_defined_symbols: <2zh>
  user_defined_symbols: <2ar>
  required_chars: 
  byte_fallback: 1
  vocabulary_output_piece_score: 1
  train_extremely_large_corpus: 0
  seed_sentencepieces_file: 
  hard_vocab_limit: 1
  use_all_vocab: 0
  unk_id: 1
  bos_id: 2
  eos_id: 3
  


chars/token on FLORES dev:
  en: 3.91
  hi: 3.53
  zh: 1.50
  ar: 3.39

FLORES dev[0] pieces:
  en: ['▁On', '▁Mon', 'day', ',', '▁scientists', '▁from', '▁the', '▁Stan', 'ford', '▁University', '▁School', '▁of', '▁Medicine', '▁announced', '▁the', '▁in', 'ven', 'tion', '▁of', '▁a', '▁new', '▁', 'dia', 'gnostic', '▁to', 'ol', '▁that', '▁can', '▁sort', '▁cells', '▁by', '▁type', ':', '▁a', '▁', 'tin', 'y', '▁print', 'able', '▁ch', 'ip', '▁that', '▁can', '▁be', '▁manufacture', 'd', '▁using', '▁standard', '▁in', 'k', 'j', 'et', '▁print', 'ers', '▁for', '▁', 'pos', 's', 'ib', 'ly', '▁about', '▁one', '▁U', '.', 'S', '.', '▁cent', '▁each', '.']
  hi: ['▁सोम', 'वार', '▁को', ',', '▁स्ट', 'ैन', 'फ़', 'ो', 'र्ड', '▁यूनिवर्सिटी', '▁स्कूल', '▁ऑफ़', '▁मे', 'डि', 'सि', 'न', '▁के', '▁वैज्ञानिक', 'ों', '▁ने', '▁एक', '▁नए', '▁डाय', 'ग्', 'नो', 'स्ट', 'िक', '▁उपकरण', '▁के', '▁आविष्कार', '▁की', '▁घोषणा', '▁की', '▁जो', '▁कोशिकाओं', '▁को', '▁उनके', '▁प्रकार', '▁के', '▁आधार', '▁पर', '▁छ', 'ाँ', 'ट', '▁सकता', '▁